# Multi-Object Detection on VisDrone

## 1. Project Overview

The goal of this project is to detect and classify multiple objects in aerial images from the VisDrone dataset.

We evaluate different object detection architectures and investigate how architectural changes and training configurations affect detection performance, particularly for small objects.

The experiments include:
- **RetinaNet** — one-stage anchor-based detector.
- **RetinaNet with Small Anchors** — modified anchor sizes for small-object detection.
- **RetinaNet with Modified Focal Loss** — adjusted classification loss.
- **Faster R-CNN** — two-stage anchor-based detector.

Models are evaluated using COCO-style Average Precision metrics, including mAP@50:95, AP50, and AP for different object sizes.

The final goal is to compare detection accuracy and inference speed and generate predictions for the Kaggle competition.


In [21]:
!git clone https://github.com/lilialania/drone-multi-object-detection.git

%cd drone-multi-object-detection

Cloning into 'drone-multi-object-detection'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 116 (delta 55), reused 86 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 2.05 MiB | 9.45 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/kaggle/working/drone-multi-object-detection/drone-multi-object-detection


## 2. Dataset Preparation

The VisDrone dataset contains aerial images with multiple object instances, including pedestrians and different types of vehicles.

For the experiments, we use eight object classes: person, bicycle, car, van, truck, tricycle, bus, and motor.

To reduce computational costs, we create subsets of the original training data by selecting frames at fixed intervals.

The original training set contains 24,201 images. Different experiments use different subsets, so their results should be interpreted with this limitation in mind.

The validation set is used to evaluate detection performance and compare model configurations.

In [6]:
!python -m src.make_subset \
    --data-root /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 \
    --output-dir /kaggle/working/visdrone-subset \
    --step 5 \
    --splits train val

Created 5439 frames; archive: /kaggle/working/visdrone-subset.zip


In [9]:
from huggingface_hub import HfApi
import huggingface_hub

print("Version:", huggingface_hub.__version__)
print("HfApi works:", HfApi is not None)

Version: 0.36.2
HfApi works: True


In [11]:
import os
from getpass import getpass
from huggingface_hub import HfApi

os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

api = HfApi(token=os.environ["HF_TOKEN"])

username = api.whoami()["name"]
repo_id = f"{username}/visdrone-retinanet"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True
)

os.environ["HF_REPO_ID"] = repo_id

print("Repository ready:", repo_id)

Hugging Face token:  ········


Repository ready: lilialania/visdrone-retinanet


## 3. RetinaNet Baseline

### Model Architecture

RetinaNet is a one-stage anchor-based object detector. It predicts object classes and bounding boxes directly from feature maps, without a separate region proposal stage.

The model uses a ResNet-50 backbone with a Feature Pyramid Network (FPN), which enables object detection at multiple scales.

RetinaNet also uses Focal Loss to reduce the influence of easy background examples and focus training on harder classification cases.

### Training Configuration

- Backbone: ResNet-50 FPN
- Initialization: COCO-pretrained weights
- Training epochs: 12
- Batch size: 1
- Learning rate: 0.001
- Minimum image size: 640
- Maximum image size: 1280

The baseline experiment uses a reduced training subset to limit GPU usage.

In [16]:
import os

os.environ["STOP_AFTER_EPOCH"] = "2"

!python -m src.models.retinanet train \
    --data-root /kaggle/working/visdrone-subset \
    --epochs 12 \
    --batch 1 \
    --lr 0.001 \
    --min-size 640 \
    --max-size 1280 \
    --out-dir /kaggle/working/runs/retinanet_lr001

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth
100%|█████████████████████████████████████████| 146M/146M [00:00<00:00, 223MB/s]
epoch 1/12 step 50/4868 loss 1.3869
epoch 1/12 step 100/4868 loss 1.2639
epoch 1/12 step 150/4868 loss 1.1552
epoch 1/12 step 200/4868 loss 1.0879
epoch 1/12 step 250/4868 loss 1.0607
epoch 1/12 step 300/4868 loss 1.0320
epoch 1/12 step 350/4868 loss 0.9979
epoch 1/12 step 400/4868 loss 0.9637
epoch 1/12 step 450/4868 loss 0.9461
epoch 1/12 step 500/4868 loss 0.9288
epoch 1/12 step 550/4868 loss 0.9065
epoch 1/12 step 600/4868 loss 0.8923
epoch 1/12 step 650/4868 loss 0.8797
epoch 1/12 step 700/4868 loss 0.8668
epoch 1/12 step 750/4868 loss 0.8551
epoch 1/12 step 800/4868 loss 0.8497
epoch 1/12 step 850/4868 loss 0.8412
epoch 1/12 step 900/4868 loss 0.8340
epoch 1/12 step 950/4868 loss 0.8255
epoch 1/12 step 1000/4868 loss 0.8163
ep

In [7]:
import os

os.environ["STOP_AFTER_EPOCH"] = "2"

!python -m src.models.retinanet train \
    --data-root /kaggle/working/visdrone-subset \
    --epochs 12 \
    --batch 1 \
    --lr 0.001 \
    --min-size 640 \
    --max-size 1280 \
    --out-dir /kaggle/working/runs/retinanet_lr001

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth
100%|█████████████████████████████████████████| 146M/146M [00:00<00:00, 234MB/s]
epoch 1/12 step 50/4868 loss 1.3762
epoch 1/12 step 100/4868 loss 1.3658
epoch 1/12 step 150/4868 loss 1.3750
epoch 1/12 step 200/4868 loss 1.3065
epoch 1/12 step 250/4868 loss 1.2289
epoch 1/12 step 300/4868 loss 1.1687
epoch 1/12 step 350/4868 loss 1.1256
epoch 1/12 step 400/4868 loss 1.0765
epoch 1/12 step 450/4868 loss 1.0449
epoch 1/12 step 500/4868 loss 1.0107
epoch 1/12 step 550/4868 loss 0.9836
epoch 1/12 step 600/4868 loss 0.9599
epoch 1/12 step 650/4868 loss 0.9384
epoch 1/12 step 700/4868 loss 0.9256
epoch 1/12 step 750/4868 loss 0.9141
epoch 1/12 step 800/4868 loss 0.9025
epoch 1/12 step 850/4868 loss 0.8917
epoch 1/12 step 900/4868 loss 0.8795
epoch 1/12 step 950/4868 loss 0.8707
epoch 1/12 step 1000/4868 loss 0.8611
ep

In [19]:
!python -m src.models.retinanet evaluate \
    --weights /kaggle/working/runs/retinanet_lr001/last.pt \
    --split-root /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val \
    --batch 1 \
    --step 5

loading annotations into memory...
Done (t=0.47s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.34s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=19.21s).
Accumulating evaluation results...
DONE (t=1.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.147
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.282
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.136
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.042
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.134
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.310
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.088
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.314
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

### Baseline Evaluation Results

The baseline RetinaNet was evaluated on every fifth image from the validation set.

| Metric | Result |
|---|---:|
| mAP@50:95 | 0.1468 |
| AP50 | 0.2820 |
| AP_small | 0.0423 |
| AP_medium | 0.134 |
| AP_large | 0.310 |
| Inference speed | 7.9 FPS |

The model achieves higher precision on large objects than on small objects. This is an important limitation for aerial object detection, where many objects occupy only a small number of pixels.

These results serve as a reference for evaluating the modified RetinaNet configurations.

In [4]:
from src.models.retinanet_small import build_model

model = build_model(pretrained=False)

print("Small anchor sizes:")
print(model.anchor_generator.sizes)

print("Aspect ratios:")
print(model.anchor_generator.aspect_ratios)

del model

Small anchor sizes:
((16, 20, 25), (32, 40, 50), (64, 80, 101), (128, 161, 203), (256, 322, 406))
Aspect ratios:
((0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0))


## 4. RetinaNet with Small Anchors

### Motivation

Aerial images contain many small objects, such as pedestrians and vehicles viewed from a distance.

The standard RetinaNet anchor sizes may be relatively large for these objects.

In this experiment, we modify the anchor generator by introducing smaller anchor sizes while keeping the overall RetinaNet architecture.

### Anchor Configuration

Original anchor sizes:

`((32, 40, 50), (64, 80, 101), (128, 161, 203), (256, 322, 406), (512, 645, 812))`

Modified anchor sizes:

`((16, 20, 25), (32, 40, 50), (64, 80, 101), (128, 161, 203), (256, 322, 406))`

The hypothesis is that smaller anchors will improve the matching of ground-truth boxes during training and increase detection precision for small objects.

In [34]:
from pathlib import Path

file = Path("src/models/faster_rcnn.py")
code = file.read_text(encoding="utf-8")

old = """    loader = DataLoader(
        YoloDetectionDataset(train_dir, train=True), batch_size=batch, shuffle=True,
        num_workers=workers, collate_fn=_collate,
    )"""

new = """    dataset = YoloDetectionDataset(train_dir, train=True)

    # Use every 5th frame only for the Small Anchors experiment
    if out_dir.name == "retinanet_small":
        from torch.utils.data import Subset
        dataset = Subset(dataset, range(0, len(dataset), 5))
        print(f"Small Anchors training images: {len(dataset)}", flush=True)

    loader = DataLoader(
        dataset, batch_size=batch, shuffle=True,
        num_workers=workers, collate_fn=_collate,
    )"""

if old in code:
    code = code.replace(old, new, 1)
    file.write_text(code, encoding="utf-8")
    print("Dataset sampling installed!")
elif "Small Anchors training images:" in code:
    print("Dataset sampling already installed.")
else:
    print("Original code not found. No changes made.")

Dataset sampling installed!


In [35]:
import ast
from pathlib import Path
from src.models.faster_rcnn import YoloDetectionDataset, image_dirs

code = Path("src/models/faster_rcnn.py").read_text(encoding="utf-8")
ast.parse(code)

train_dir, _ = image_dirs(Path(
    "/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026"
))

dataset = YoloDetectionDataset(train_dir, train=True)
indices = range(0, len(dataset), 5)

print("Full dataset:", len(dataset))
print("Small Anchors subset:", len(indices))
print("Backup installed:", "BACKUP VERIFIED: {remote_path}" in code)
print("Sampling installed:", "Small Anchors training images:" in code)

Full dataset: 24201
Small Anchors subset: 4841
Backup installed: True
Sampling installed: True


### Training

The Small Anchors model was trained for 12 epochs on 4,841 training images, selected by sampling every fifth frame.

The experiment uses the same learning rate, batch size, and input resizing configuration as the baseline.

Only the anchor generator configuration was changed at the model level.

Checkpoints were saved after each epoch to Hugging Face for reproducibility and recovery.

In [11]:
import subprocess
import os

subprocess.run(
    [
        "python", "-u", "-m", "src.models.retinanet_small", "train",
        "--data-root",
        "/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026",
        "--epochs", "12",
        "--batch", "1",
        "--lr", "0.001",
        "--min-size", "640",
        "--max-size", "1280",
        "--out-dir", "/kaggle/working/runs/retinanet_small",
        "--workers", "2",
    ],
    env=os.environ.copy(),
    check=True,
)

epoch 1/12 step 50/4841 loss 1.6216
epoch 1/12 step 100/4841 loss 1.5267
epoch 1/12 step 150/4841 loss 1.5307
epoch 1/12 step 200/4841 loss 1.4912
epoch 1/12 step 250/4841 loss 1.4226
epoch 1/12 step 300/4841 loss 1.3791
epoch 1/12 step 350/4841 loss 1.3413
epoch 1/12 step 400/4841 loss 1.3422
epoch 1/12 step 450/4841 loss 1.3016
epoch 1/12 step 500/4841 loss 1.2683
epoch 1/12 step 550/4841 loss 1.2374
epoch 1/12 step 600/4841 loss 1.2117
epoch 1/12 step 650/4841 loss 1.1889
epoch 1/12 step 700/4841 loss 1.1648
epoch 1/12 step 750/4841 loss 1.1433
epoch 1/12 step 800/4841 loss 1.1242
epoch 1/12 step 850/4841 loss 1.1066
epoch 1/12 step 900/4841 loss 1.0892
epoch 1/12 step 950/4841 loss 1.0757
epoch 1/12 step 1000/4841 loss 1.0750
epoch 1/12 step 1050/4841 loss 1.0625
epoch 1/12 step 1100/4841 loss 1.0499
epoch 1/12 step 1150/4841 loss 1.0368
epoch 1/12 step 1200/4841 loss 1.0253
epoch 1/12 step 1250/4841 loss 1.0149
epoch 1/12 step 1300/4841 loss 1.0051
epoch 1/12 step 1350/4841 loss 0

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   4%|▎         | 10.9MB /  291MB, 1.07MB/s  
New Data Upload               :  16%|█▌        | 10.5MB / 67.1MB, 1.03MB/s  

Processing Files (0 / 1)      :  11%|█         | 32.5MB /  291MB, 3.17MB/s  
New Data Upload               :  48%|████▊     | 32.2MB / 67.1MB, 3.14MB/s  

Processing Files (0 / 1)      :  19%|█▉        | 56.3MB /  291MB, 5.47MB/s 

BACKUP VERIFIED: small_anchors/epoch_01.pt
epoch 2/12 step 50/4841 loss 0.6000
epoch 2/12 step 100/4841 loss 0.6080
epoch 2/12 step 150/4841 loss 0.6182
epoch 2/12 step 200/4841 loss 0.6137
epoch 2/12 step 250/4841 loss 0.6499
epoch 2/12 step 300/4841 loss 0.6435
epoch 2/12 step 350/4841 loss 0.6343
epoch 2/12 step 400/4841 loss 0.6318
epoch 2/12 step 450/4841 loss 0.6295
epoch 2/12 step 500/4841 loss 0.6268
epoch 2/12 step 550/4841 loss 0.6248
epoch 2/12 step 600/4841 loss 0.6229
epoch 2/12 step 650/4841 loss 0.6210
epoch 2/12 step 700/4841 loss 0.6175
epoch 2/12 step 750/4841 loss 0.6152
epoch 2/12 step 800/4841 loss 0.6145
epoch 2/12 step 850/4841 loss 0.6146
epoch 2/12 step 900/4841 loss 0.6131
epoch 2/12 step 950/4841 loss 0.6128
epoch 2/12 step 1000/4841 loss 0.6109
epoch 2/12 step 1050/4841 loss 0.6109
epoch 2/12 step 1100/4841 loss 0.6091
epoch 2/12 step 1150/4841 loss 0.6078
epoch 2/12 step 1200/4841 loss 0.6061
epoch 2/12 step 1250/4841 loss 0.6052
epoch 2/12 step 1300/4841 l

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          | 1.45MB /  291MB,  141kB/s  
New Data Upload               :   2%|▏         | 1.11MB / 67.0MB,  109kB/s  

Processing Files (0 / 1)      :   7%|▋         | 21.4MB /  291MB, 2.10MB/s  
New Data Upload               :  31%|███▏      | 21.1MB / 67.0MB, 2.07MB/s  

Processing Files (0 / 1)      :  14%|█▍        | 41.4MB /  291MB, 4.04MB/s 

BACKUP VERIFIED: small_anchors/epoch_02.pt
epoch 3/12 step 50/4841 loss 0.5149
epoch 3/12 step 100/4841 loss 0.5139
epoch 3/12 step 150/4841 loss 0.5276
epoch 3/12 step 200/4841 loss 0.5333
epoch 3/12 step 250/4841 loss 0.5332
epoch 3/12 step 300/4841 loss 0.5364
epoch 3/12 step 350/4841 loss 0.5343
epoch 3/12 step 400/4841 loss 0.5340
epoch 3/12 step 450/4841 loss 0.5352
epoch 3/12 step 500/4841 loss 0.5367
epoch 3/12 step 550/4841 loss 0.5375
epoch 3/12 step 600/4841 loss 0.5356
epoch 3/12 step 650/4841 loss 0.5356
epoch 3/12 step 700/4841 loss 0.5350
epoch 3/12 step 750/4841 loss 0.5346
epoch 3/12 step 800/4841 loss 0.5338
epoch 3/12 step 850/4841 loss 0.5320
epoch 3/12 step 900/4841 loss 0.5320
epoch 3/12 step 950/4841 loss 0.5327
epoch 3/12 step 1000/4841 loss 0.5310
epoch 3/12 step 1050/4841 loss 0.5432
epoch 3/12 step 1100/4841 loss 0.5435
epoch 3/12 step 1150/4841 loss 0.5417
epoch 3/12 step 1200/4841 loss 0.5411
epoch 3/12 step 1250/4841 loss 0.5397
epoch 3/12 step 1300/4841 l

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   3%|▎         | 8.65MB /  291MB,  850kB/s  
New Data Upload               :  12%|█▏        | 8.31MB / 67.1MB,  818kB/s  

Processing Files (0 / 1)      :  12%|█▏        | 33.6MB /  291MB, 3.29MB/s  
New Data Upload               :  50%|████▉     | 33.2MB / 67.1MB, 3.26MB/s  

Processing Files (0 / 1)      :  20%|██        | 59.1MB /  291MB, 5.75MB/s 

... (14 outputs left out)

BACKUP VERIFIED: small_anchors/epoch_11.pt
epoch 12/12 step 50/4841 loss 0.3573
epoch 12/12 step 100/4841 loss 0.3583
epoch 12/12 step 150/4841 loss 0.3621
epoch 12/12 step 200/4841 loss 0.3612
epoch 12/12 step 250/4841 loss 0.3636
epoch 12/12 step 300/4841 loss 0.3598
epoch 12/12 step 350/4841 loss 0.3634
epoch 12/12 step 400/4841 loss 0.3640
epoch 12/12 step 450/4841 loss 0.3647
epoch 12/12 step 500/4841 loss 0.3648
epoch 12/12 step 550/4841 loss 0.3646
epoch 12/12 step 600/4841 loss 0.3651
epoch 12/12 step 650/4841 loss 0.3626
epoch 12/12 step 700/4841 loss 0.3631
epoch 12/12 step 750/4841 loss 0.3623
epoch 12/12 step 800/4841 loss 0.3621
epoch 12/12 step 850/4841 loss 0.3616
epoch 12/12 step 900/4841 loss 0.3618
epoch 12/12 step 950/4841 loss 0.3619
epoch 12/12 step 1000/4841 loss 0.3628
epoch 12/12 step 1050/4841 loss 0.3625
epoch 12/12 step 1100/4841 loss 0.3635
epoch 12/12 step 1150/4841 loss 0.3648
epoch 12/12 step 1200/4841 loss 0.3641
epoch 12/12 step 1250/4841 loss 0.3648
ep

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_small/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   1%|          | 2.00MB /  291MB,  196kB/s  
New Data Upload               :   2%|▏         | 1.66MB / 67.0MB,  164kB/s  

Processing Files (0 / 1)      :   9%|▉         | 25.9MB /  291MB, 2.54MB/s  
New Data Upload               :  38%|███▊      | 25.5MB / 67.0MB, 2.50MB/s  

Processing Files (0 / 1)      :  21%|██        | 60.8MB /  291MB, 5.93MB/s 

BACKUP VERIFIED: small_anchors/epoch_12.pt


CompletedProcess(args=['python', '-u', '-m', 'src.models.retinanet_small', 'train', '--data-root', '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026', '--epochs', '12', '--batch', '1', '--lr', '0.001', '--min-size', '640', '--max-size', '1280', '--out-dir', '/kaggle/working/runs/retinanet_small', '--workers', '2'], returncode=0)

In [12]:
import subprocess
from pathlib import Path

weights = "/kaggle/working/runs/retinanet_small/last.pt"

assert Path(weights).is_file(), "Checkpoint not found"

subprocess.run(
    [
        "python", "-u", "-m", "src.models.retinanet_small", "evaluate",
        "--weights", weights,
        "--split-root",
        "/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val",
        "--batch", "1",
        "--step", "5",
    ],
    check=True,
)

loading annotations into memory...
Done (t=0.59s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.40s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=36.02s).
Accumulating evaluation results...
DONE (t=1.53s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.145
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.315
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.117
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.061
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.136
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.278
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.090
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.343
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

CompletedProcess(args=['python', '-u', '-m', 'src.models.retinanet_small', 'evaluate', '--weights', '/kaggle/working/runs/retinanet_small/last.pt', '--split-root', '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val', '--batch', '1', '--step', '5'], returncode=0)

### Evaluation Results

| Metric | Baseline | Small Anchors |
|---|---:|---:|
| mAP@50:95 | 0.1468 | 0.1459 |
| AP50 | 0.2820 | 0.3151 |
| AP_small | 0.0423 | 0.0608 |
| AP_medium | 0.134 | 0.136 |
| AP_large | 0.310 | 0.278 |

### Interpretation

The Small Anchors configuration improved AP_small from 0.0423 to 0.0608, representing an approximately 44% relative increase.

AP50 also increased, while overall mAP@50:95 remained nearly unchanged.

This suggests that smaller anchors may be useful for detecting small objects, although the models were trained on different subsets and the improvement cannot be attributed exclusively to anchor size.

In [14]:

from pathlib import Path
import os

source = Path(
    "/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026"
)

output = Path("/kaggle/working/visdrone-step20")

images = sorted((source / "train" / "images").rglob("*.jpg"))
selected = images[::20]

print("Full dataset:", len(images))
print("Selected images:", len(selected))

for image in selected:
    relative = image.relative_to(source / "train" / "images")

    target_image = output / "train" / "images" / relative
    target_label = (
        output / "train" / "labels" / relative
    ).with_suffix(".txt")

    source_label = (
        source / "train" / "labels" / relative
    ).with_suffix(".txt")

    target_image.parent.mkdir(parents=True, exist_ok=True)
    target_label.parent.mkdir(parents=True, exist_ok=True)

    # Symlinks instead of copying images
    if not target_image.exists():
        os.symlink(image, target_image)

    if source_label.exists() and not target_label.exists():
        os.symlink(source_label, target_label)

print("Preprocessing complete!")


Full dataset: 24201
Selected images: 1211
Preprocessing complete!


## 5. RetinaNet with Modified Focal Loss

### Motivation

RetinaNet uses Focal Loss to address class imbalance between background and foreground examples.

This experiment investigates whether changing the classification weighting parameter improves object detection performance.

We use the Small Anchors model and modify the Focal Loss parameters:

- Alpha: 0.5
- Gamma: 2.0

The original configuration uses alpha = 0.25 and gamma = 2.0.

### Training Setup

To reduce computational cost, the training dataset was further reduced to 1,211 images by selecting every twentieth frame.

The model was trained for 6 epochs with a learning rate of 0.001 and batch size 1.

In [30]:
from pathlib import Path
import ast

file = Path("src/models/faster_rcnn.py")
code = file.read_text(encoding="utf-8")

target = '        print(f"epoch {epoch + 1} done, mean loss {running / max(len(loader), 1):.4f}, saved {checkpoint}")'

backup = '''
        # Automatically back up Focal Loss checkpoints
        if out_dir.name == "retinanet_focal":
            import os
            from huggingface_hub import HfApi

            api = HfApi(token=os.environ["HF_TOKEN"])
            remote_path = f"focal_alpha05_step20/epoch_{epoch + 1:02d}.pt"

            print(f"Uploading checkpoint: {remote_path}", flush=True)

            api.upload_file(
                path_or_fileobj=str(checkpoint),
                path_in_repo=remote_path,
                repo_id="lilialania/visdrone-retinanet",
                repo_type="model",
                commit_message=f"Focal Loss epoch {epoch + 1}",
            )

            if not api.file_exists(
                repo_id="lilialania/visdrone-retinanet",
                filename=remote_path,
                repo_type="model",
            ):
                raise RuntimeError("Backup verification failed")

            print(f"BACKUP VERIFIED: {remote_path}", flush=True)
'''

if "BACKUP VERIFIED: {remote_path}" in code:
    print("Backup code already exists. No changes made.")
elif code.count(target) != 1:
    raise RuntimeError("Checkpoint-saving line not found. No changes made.")
else:
    updated = code.replace(target, target + "\n" + backup, 1)
    ast.parse(updated)
    file.write_text(updated, encoding="utf-8")
    print("Focal Loss automatic backup installed!")

Focal Loss automatic backup installed!


In [32]:
import os
import subprocess
from pathlib import Path

data_root = "/kaggle/working/visdrone-step20"
out_dir = "/kaggle/working/runs/retinanet_focal"

checkpoint = Path(out_dir) / "last.pt"

command = [
    "python", "-u", "-m", "src.models.retinanet_focal", "train",
    "--data-root", data_root,
    "--epochs", "6",
    "--batch", "1",
    "--lr", "0.001",
    "--min-size", "640",
    "--max-size", "1280",
    "--out-dir", out_dir,
    "--workers", "2",
]

if checkpoint.exists():
    command.append("--resume")
    print("Resuming from checkpoint")
else:
    print("Starting new Focal Loss experiment")

subprocess.run(
    command,
    env=os.environ.copy(),
    check=True,
)

Starting new Focal Loss experiment
epoch 1/6 step 50/1211 loss 2.7828
epoch 1/6 step 100/1211 loss 2.5615
epoch 1/6 step 150/1211 loss 2.5899
epoch 1/6 step 200/1211 loss 2.5355
epoch 1/6 step 250/1211 loss 2.4326
epoch 1/6 step 300/1211 loss 2.4087
epoch 1/6 step 350/1211 loss 2.4423
epoch 1/6 step 400/1211 loss 2.4556
epoch 1/6 step 450/1211 loss 2.4107
epoch 1/6 step 500/1211 loss 2.3363
epoch 1/6 step 550/1211 loss 2.2578
epoch 1/6 step 600/1211 loss 2.1709
epoch 1/6 step 650/1211 loss 2.1132
epoch 1/6 step 700/1211 loss 2.0511
epoch 1/6 step 750/1211 loss 1.9954
epoch 1/6 step 800/1211 loss 1.9487
epoch 1/6 step 850/1211 loss 1.9185
epoch 1/6 step 900/1211 loss 1.8829
epoch 1/6 step 950/1211 loss 1.8473
epoch 1/6 step 1000/1211 loss 1.8142
epoch 1/6 step 1050/1211 loss 1.7804
epoch 1/6 step 1100/1211 loss 1.7507
epoch 1/6 step 1150/1211 loss 1.7253
epoch 1/6 step 1200/1211 loss 1.7029
epoch 1 done, mean loss 1.6982, saved /kaggle/working/runs/retinanet_focal/last.pt
Uploading chec

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   2%|▏         | 4.77MB /  291MB,  469kB/s  
New Data Upload               :   7%|▋         | 4.44MB / 67.1MB,  437kB/s  

Processing Files (0 / 1)      :   7%|▋         | 21.4MB /  291MB, 2.10MB/s  
New Data Upload               :  31%|███▏      | 21.1MB / 67.1MB, 2.07MB/s  

Processing Files (0 / 1)      :  13%|█▎        | 38.1MB /  291MB, 3.71MB/s 

BACKUP VERIFIED: focal_alpha05_step20/epoch_01.pt
epoch 2/6 step 50/1211 loss 1.1345
epoch 2/6 step 100/1211 loss 1.1160
epoch 2/6 step 150/1211 loss 1.1196
epoch 2/6 step 200/1211 loss 1.0832
epoch 2/6 step 250/1211 loss 1.0660
epoch 2/6 step 300/1211 loss 1.0555
epoch 2/6 step 350/1211 loss 1.0415
epoch 2/6 step 400/1211 loss 1.0308
epoch 2/6 step 450/1211 loss 1.0264
epoch 2/6 step 500/1211 loss 1.0295
epoch 2/6 step 550/1211 loss 1.0272
epoch 2/6 step 600/1211 loss 1.0231
epoch 2/6 step 650/1211 loss 1.0155
epoch 2/6 step 700/1211 loss 1.0109
epoch 2/6 step 750/1211 loss 1.0077
epoch 2/6 step 800/1211 loss 1.0001
epoch 2/6 step 850/1211 loss 0.9945
epoch 2/6 step 900/1211 loss 0.9933
epoch 2/6 step 950/1211 loss 0.9926
epoch 2/6 step 1000/1211 loss 0.9928
epoch 2/6 step 1050/1211 loss 0.9884
epoch 2/6 step 1100/1211 loss 0.9820
epoch 2/6 step 1150/1211 loss 0.9778
epoch 2/6 step 1200/1211 loss 0.9723
epoch 2 done, mean loss 0.9708, saved /kaggle/working/runs/retinanet_focal/last.pt

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          | 1.44MB /  291MB,  141kB/s  
New Data Upload               :   2%|▏         | 1.11MB / 67.1MB,  109kB/s  

Processing Files (0 / 1)      :   4%|▍         | 12.0MB /  291MB, 1.17MB/s  
New Data Upload               :  17%|█▋        | 11.6MB / 67.1MB, 1.14MB/s  

Processing Files (0 / 1)      :  18%|█▊        | 52.4MB /  291MB, 5.12MB/s 

BACKUP VERIFIED: focal_alpha05_step20/epoch_02.pt
epoch 3/6 step 50/1211 loss 0.8453
epoch 3/6 step 100/1211 loss 0.8527
epoch 3/6 step 150/1211 loss 0.8414
epoch 3/6 step 200/1211 loss 0.8341
epoch 3/6 step 250/1211 loss 0.8407
epoch 3/6 step 300/1211 loss 0.8414
epoch 3/6 step 350/1211 loss 0.8405
epoch 3/6 step 400/1211 loss 0.8457
epoch 3/6 step 450/1211 loss 0.8477
epoch 3/6 step 500/1211 loss 0.8495
epoch 3/6 step 550/1211 loss 0.8443
epoch 3/6 step 600/1211 loss 0.8417
epoch 3/6 step 650/1211 loss 0.8398
epoch 3/6 step 700/1211 loss 0.8379
epoch 3/6 step 750/1211 loss 0.8358
epoch 3/6 step 800/1211 loss 0.8396
epoch 3/6 step 850/1211 loss 0.8379
epoch 3/6 step 900/1211 loss 0.8370
epoch 3/6 step 950/1211 loss 0.8341
epoch 3/6 step 1000/1211 loss 0.8306
epoch 3/6 step 1050/1211 loss 0.8313
epoch 3/6 step 1100/1211 loss 0.8293
epoch 3/6 step 1150/1211 loss 0.8274
epoch 3/6 step 1200/1211 loss 0.8291
epoch 3 done, mean loss 0.8284, saved /kaggle/working/runs/retinanet_focal/last.pt

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          | 1.44MB /  291MB,  141kB/s  
New Data Upload               :   2%|▏         | 1.11MB / 67.1MB,  109kB/s  

Processing Files (0 / 1)      :   6%|▌         | 17.0MB /  291MB, 1.66MB/s  
New Data Upload               :  25%|██▍       | 16.6MB / 67.1MB, 1.63MB/s  

Processing Files (0 / 1)      :  23%|██▎       | 66.8MB /  291MB, 6.53MB/s 

... (4 outputs left out)

BACKUP VERIFIED: focal_alpha05_step20/epoch_05.pt
epoch 6/6 step 50/1211 loss 0.7039
epoch 6/6 step 100/1211 loss 0.6973
epoch 6/6 step 150/1211 loss 0.6884
epoch 6/6 step 200/1211 loss 0.6947
epoch 6/6 step 250/1211 loss 0.6899
epoch 6/6 step 300/1211 loss 0.6970
epoch 6/6 step 350/1211 loss 0.6966
epoch 6/6 step 400/1211 loss 0.6941
epoch 6/6 step 450/1211 loss 0.6985
epoch 6/6 step 500/1211 loss 0.7003
epoch 6/6 step 550/1211 loss 0.6986
epoch 6/6 step 600/1211 loss 0.6974
epoch 6/6 step 650/1211 loss 0.6982
epoch 6/6 step 700/1211 loss 0.6967
epoch 6/6 step 750/1211 loss 0.6953
epoch 6/6 step 800/1211 loss 0.6935
epoch 6/6 step 850/1211 loss 0.6925
epoch 6/6 step 900/1211 loss 0.6922
epoch 6/6 step 950/1211 loss 0.6913
epoch 6/6 step 1000/1211 loss 0.6909
epoch 6/6 step 1050/1211 loss 0.6914
epoch 6/6 step 1100/1211 loss 0.6904
epoch 6/6 step 1150/1211 loss 0.6889
epoch 6/6 step 1200/1211 loss 0.6873
epoch 6 done, mean loss 0.6867, saved /kaggle/working/runs/retinanet_focal/last.pt

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  336kB /  291MB,   ???B/s  

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

  ...s/retinanet_focal/last.pt:   0%|          |  336kB /  291MB            

Processing Files (0 / 1)      :   0%|          |  891kB /  291MB, 86.6kB/s  
New Data Upload               :   1%|          |  555kB / 67.1MB, 54.6kB/s  

Processing Files (0 / 1)      :  10%|▉         | 28.6MB /  291MB, 2.81MB/s  
New Data Upload               :  42%|████▏     | 28.3MB / 67.1MB, 2.78MB/s  

Processing Files (0 / 1)      :  19%|█▉        | 56.4MB /  291MB, 5.50MB/s 

BACKUP VERIFIED: focal_alpha05_step20/epoch_06.pt


CompletedProcess(args=['python', '-u', '-m', 'src.models.retinanet_focal', 'train', '--data-root', '/kaggle/working/visdrone-step20', '--epochs', '6', '--batch', '1', '--lr', '0.001', '--min-size', '640', '--max-size', '1280', '--out-dir', '/kaggle/working/runs/retinanet_focal', '--workers', '2'], returncode=0)

In [35]:
import subprocess

command = [
    "python", "-u", "-m", "src.models.retinanet_focal",
    "evaluate",
    "--weights",
    "/kaggle/working/runs/retinanet_focal/last.pt",
    "--split-root",
    "/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val",
    "--batch", "1",
    "--step", "5",
]

subprocess.run(command, check=True)

loading annotations into memory...
Done (t=0.47s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.45s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=38.97s).
Accumulating evaluation results...
DONE (t=2.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.092
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.210
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.068
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.045
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.084
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.195
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.036
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.304
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

CompletedProcess(args=['python', '-u', '-m', 'src.models.retinanet_focal', 'evaluate', '--weights', '/kaggle/working/runs/retinanet_focal/last.pt', '--split-root', '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val', '--batch', '1', '--step', '5'], returncode=0)

### Evaluation Results

| Metric | Result |
|---|---:|
| mAP@50:95 | 0.0928 |
| AP50 | 0.2101 |
| AP_small | 0.0445 |
| AP_medium | 0.084 |
| AP_large | 0.195 |
| Inference speed | 8.0 FPS |

### Interpretation

The modified Focal Loss configuration achieved a lower overall mAP than the previous RetinaNet experiments.

However, this experiment also used fewer training images and fewer epochs. Therefore, the results do not demonstrate that alpha = 0.5 is worse than alpha = 0.25.

A controlled comparison would require training both configurations on the same dataset with identical training settings.

## 6. Faster R-CNN

### Model Architecture

Faster R-CNN is a two-stage anchor-based object detector.

Unlike RetinaNet, Faster R-CNN separates detection into two stages:

1. **Region Proposal Network (RPN):** generates candidate regions that may contain objects.
2. **RoI Heads:** classify the proposed regions and refine their bounding boxes.

The model uses a ResNet-50 FPN backbone and COCO-pretrained weights.

### Training Configuration

- Training images: 1,211
- Epochs: 6
- Batch size: 1
- Learning rate: 0.001
- Minimum image size: 640
- Maximum image size: 1280

Checkpoints were saved after each epoch to Hugging Face.

The objective is to compare a two-stage detector with the one-stage RetinaNet models in terms of accuracy and inference speed.

In [42]:
command = [
    "python", "-u", "-m", "src.models.faster_rcnn", "train",
    "--data-root", "/kaggle/working/visdrone-step20",
    "--epochs", "6",
    "--batch", "1",
    "--lr", "0.001",
    "--min-size", "640",
    "--max-size", "1280",
    "--out-dir", str(out_dir),
    "--workers", "2",
]

# Відновлюємо навчання, якщо checkpoint уже існує
if (out_dir / "last.pt").exists():
    command.append("--resume")
    print("Resuming Faster R-CNN training")
else:
    print("Starting Faster R-CNN training")

train_with_backup(command)

Starting Faster R-CNN training
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth

  0%|          | 0.00/167M [00:00<?, ?B/s]
  7%|▋         | 12.5M/167M [00:00<00:01, 128MB/s]
 22%|██▏       | 37.0M/167M [00:00<00:00, 203MB/s]
 37%|███▋      | 61.4M/167M [00:00<00:00, 227MB/s]
 51%|█████▏    | 85.9M/167M [00:00<00:00, 238MB/s]
 66%|██████▌   | 110M/167M [00:00<00:00, 245MB/s] 
 81%|████████  | 135M/167M [00:00<00:00, 250MB/s]
 96%|█████████▌| 160M/167M [00:00<00:00, 253MB/s]
100%|██████████| 167M/167M [00:00<00:00, 239MB/s]
epoch 1/6 step 50/1211 loss 1.3363
epoch 1/6 step 100/1211 loss 1.0908
epoch 1/6 step 150/1211 loss 0.9972
epoch 1/6 step 200/1211 loss 0.9728
epoch 1/6 step 250/1211 loss 0.9694
epoch 1/6 step 300/1211 loss 0.9461
epoch 1/6 step 350/1211 loss 0.9311
epoch 1/6 step 400/1211 loss 0.9239
epoch 1/6 step 450/1211 loss 0.9213
epoch 1/6 step 500

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

BACKUP VERIFIED: faster_rcnn_step20/epoch_01.pt
epoch 2/6 step 50/1211 loss 0.7993
epoch 2/6 step 100/1211 loss 0.8162
epoch 2/6 step 150/1211 loss 0.8217
epoch 2/6 step 200/1211 loss 0.8073
epoch 2/6 step 250/1211 loss 0.7934
epoch 2/6 step 300/1211 loss 0.7964
epoch 2/6 step 350/1211 loss 0.7900
epoch 2/6 step 400/1211 loss 0.7941
epoch 2/6 step 450/1211 loss 0.7949
epoch 2/6 step 500/1211 loss 0.7970
epoch 2/6 step 550/1211 loss 0.8000
epoch 2/6 step 600/1211 loss 0.7987
epoch 2/6 step 650/1211 loss 0.7990
epoch 2/6 step 700/1211 loss 0.7950
epoch 2/6 step 750/1211 loss 0.7974
epoch 2/6 step 800/1211 loss 0.7972
epoch 2/6 step 850/1211 loss 0.7926
epoch 2/6 step 900/1211 loss 0.7912
epoch 2/6 step 950/1211 loss 0.7907
epoch 2/6 step 1000/1211 loss 0.7896
epoch 2/6 step 1050/1211 loss 0.7911
epoch 2/6 step 1100/1211 loss 0.7916
epoch 2/6 step 1150/1211 loss 0.7886
epoch 2/6 step 1200/1211 loss 0.7915
epoch 2 done, mean loss 0.7904, saved /kaggle/working/runs/faster_rcnn_step20/last.p

Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (15 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

BACKUP VERIFIED: faster_rcnn_step20/epoch_06.pt


In [43]:
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ["HF_TOKEN"])

repo_id = "lilialania/visdrone-retinanet"

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="model"
)

experiments = [
    "small_anchors/",
    "focal_alpha05_step20/",
    "faster_rcnn_step20/",
]

for experiment in experiments:
    checkpoints = sorted(
        f for f in files
        if f.startswith(experiment) and f.endswith(".pt")
    )

    print(f"\n{experiment}")
    print(f"Saved checkpoints: {len(checkpoints)}")
    print("Latest:", checkpoints[-1] if checkpoints else "NOT FOUND")


small_anchors/
Saved checkpoints: 12
Latest: small_anchors/epoch_12.pt

focal_alpha05_step20/
Saved checkpoints: 6
Latest: focal_alpha05_step20/epoch_06.pt

faster_rcnn_step20/
Saved checkpoints: 6
Latest: faster_rcnn_step20/epoch_06.pt


In [44]:
from pathlib import Path
import json

summary = {
    "dataset": "VisDrone",
    "validation_step": 5,
    "experiments": {
        "retinanet_baseline": {
            "epochs": 12,
            "train_images": 24201,
            "map": 0.1468,
            "ap50": 0.2820,
            "checkpoint": "checkpoints/epoch_12.pt"
        },
        "retinanet_small_anchors": {
            "epochs": 12,
            "train_images": 4841,
            "map": 0.1459,
            "ap50": 0.3151,
            "checkpoint": "small_anchors/epoch_12.pt"
        },
        "retinanet_focal": {
            "epochs": 6,
            "train_images": 1211,
            "alpha": 0.5,
            "gamma": 2.0,
            "map": 0.0928,
            "ap50": 0.2101,
            "checkpoint": "focal_alpha05_step20/epoch_06.pt"
        },
        "faster_rcnn": {
            "epochs": 6,
            "train_images": 1211,
            "checkpoint": "faster_rcnn_step20/epoch_06.pt",
            "validation": "not evaluated yet"
        }
    }
}

summary_path = Path("/kaggle/working/experiment_summary.json")
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

api.upload_file(
    path_or_fileobj=str(summary_path),
    path_in_repo="experiment_summary.json",
    repo_id=repo_id,
    repo_type="model"
)

print("Experiment summary saved to Hugging Face!")

Experiment summary saved to Hugging Face!


In [4]:
import subprocess
import sys

command = [
    sys.executable, "-u", "-m", "src.models.faster_rcnn",
    "evaluate",
    "--weights", str(checkpoint),
    "--split-root",
    "/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val",
    "--batch", "1",
    "--step", "5",
]

subprocess.run(command, check=True)

loading annotations into memory...
Done (t=0.67s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.47s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=43.02s).
Accumulating evaluation results...
DONE (t=1.99s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.167
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.361
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.132
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.087
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.165
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.301
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.092
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.391
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'src.models.faster_rcnn', 'evaluate', '--weights', '/root/.cache/huggingface/hub/models--lilialania--visdrone-retinanet/snapshots/6759be09efe6a86982c6e504d242f22229d9cd6a/faster_rcnn_step20/epoch_06.pt', '--split-root', '/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/val', '--batch', '1', '--step', '5'], returncode=0)

### Evaluation Results

| Metric | Result |
|---|---:|
| mAP@50:95 | 0.1675 |
| AP50 | 0.3611 |
| AP_small | 0.0873 |
| AP_medium | 0.165 |
| AP_large | 0.301 |
| Inference speed | 4.1 FPS |

### Interpretation

Faster R-CNN achieved the highest validation mAP among the evaluated configurations.

It also achieved the highest AP_small, indicating better detection performance on small objects in the evaluated subset.

However, its inference speed was approximately half that of RetinaNet.

This demonstrates a trade-off between detection accuracy and computational efficiency.

Since the experiments used different training subsets and training durations, the results should not be interpreted as a fully controlled architecture comparison.

## 7. Kaggle Test Inference and Submission

After evaluating the models on the validation set, Faster R-CNN was selected for the first Kaggle submission because it achieved the highest validation mAP.

The model was used to generate predictions for all 3,811 test images.

Each prediction contains:

`class_id confidence x y width height`

During submission validation, nine invalid bounding boxes with non-positive dimensions were detected and removed.

The final CSV contains one row per test image with columns `image_id` and `PredictionString`.

### Kaggle Result

**Public Score: 0.1426625907**

The public score was lower than the validation mAP of 0.1675. Possible reasons include differences between validation and test images, the evaluation setup, and model generalization.

This result provides a starting point for further inference improvements, including tiled inference and postprocessing.

In [14]:
import pandas as pd

submission = pd.read_csv("/kaggle/working/submission_retinanet.csv")
test = pd.read_csv(f"{test_root}/test_images.csv")

print("Submission rows:", len(submission))
print("Test rows:", len(test))
print("Unique image IDs:", submission["image_id"].nunique())
print("Missing IDs:", len(set(test["image_id"]) - set(submission["image_id"])))
print("Null predictions:", submission["PredictionString"].isna().sum())

# Кожне передбачення має 6 значень:
# class_id, confidence, x, y, width, height
lengths = submission["PredictionString"].dropna().apply(
    lambda x: len(str(x).split())
)

print("Valid prediction lengths:", (lengths % 6 == 0).all())

Submission rows: 579
Test rows: 3811
Unique image IDs: 579
Missing IDs: 3232
Null predictions: 0
Valid prediction lengths: True


In [5]:
import subprocess
import sys
from pathlib import Path

output = Path("/kaggle/working/submission_faster_rcnn.csv")

command = [
    sys.executable, "-u", "-m", "src.models.faster_rcnn",
    "predict",
    "--weights", str(weights_path),
    "--test-root", str(data_root),
    "--out", str(output),
    "--batch", "1",
]

subprocess.run(command, check=True)

print("\nSubmission created:", output.exists())

if output.exists():
    print("Size:", round(output.stat().st_size / 1024**2, 2), "MB")

Wrote 3811 rows to /kaggle/working/submission_faster_rcnn.csv

Submission created: True
Size: 34.06 MB


In [6]:
from huggingface_hub import HfApi
from pathlib import Path
import os

submission_path = Path("/kaggle/working/submission_faster_rcnn.csv")

assert submission_path.exists(), "Submission file not found!"

api = HfApi(token=os.environ["HF_TOKEN"])

api.upload_file(
    path_or_fileobj=str(submission_path),
    path_in_repo="submissions/submission_faster_rcnn.csv",
    repo_id="lilialania/visdrone-retinanet",
    repo_type="model",
    commit_message="Save Faster R-CNN Kaggle submission"
)

verified = api.file_exists(
    repo_id="lilialania/visdrone-retinanet",
    filename="submissions/submission_faster_rcnn.csv",
    repo_type="model"
)

print("BACKUP VERIFIED:", verified)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

BACKUP VERIFIED: True


In [7]:
import pandas as pd
from pathlib import Path

data_root = Path(
    "/kaggle/input/competitions/"
    "multiple-object-video-detection-hw-2-ucu-ai-2026"
)

sub = pd.read_csv(
    "/kaggle/working/submission_faster_rcnn.csv"
)

test = pd.read_csv(data_root / "test_images.csv")

print("Submission shape:", sub.shape)
print("Submission columns:", sub.columns.tolist())
print("Test columns:", test.columns.tolist())

print("\nFirst 3 predictions:")
print(sub.head(3).to_string(index=False, max_colwidth=100))

print("\nMissing values:")
print(sub.isna().sum())

print("\nExpected test images:", len(test))
print("Submission rows:", len(sub))

Submission shape: (3811, 2)
Submission columns: ['image_id', 'PredictionString']
Test columns: ['image_id', 'file_name', 'clip', 'frame', 'width', 'height']

First 3 predictions:
        image_id                                                                                     PredictionString
clip_001_0000001 0 0.9871 1714.8 788.6 40.7 145.1 0 0.9871 728.5 896.6 59.9 160.0 0 0.9814 732.2 758.3 39.7 115.9 ...
clip_001_0000002 0 0.9849 1732.9 790.5 43.0 145.5 0 0.9843 727.1 897.9 61.1 164.4 0 0.9803 623.7 925.0 61.0 150.8 ...
clip_001_0000003 0 0.9908 1251.4 781.5 37.0 110.1 0 0.9823 774.1 900.8 61.1 160.1 0 0.9821 618.8 938.8 59.6 133.6 ...

Missing values:
image_id            0
PredictionString    0
dtype: int64

Expected test images: 3811
Submission rows: 3811


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

clean_path = Path("/kaggle/working/submission_faster_rcnn_clean.csv")
test_csv = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/test_images.csv")

sub = pd.read_csv(clean_path, keep_default_na=False)
test = pd.read_csv(test_csv)

print("Submission shape:", sub.shape)
print("Unique image ids:", sub["image_id"].nunique())
print("Expected test rows:", len(test))
print("Missing IDs:", len(set(test["image_id"]) - set(sub["image_id"])))
print("Extra IDs:", len(set(sub["image_id"]) - set(test["image_id"])))

valid = True
bad_reason = None

for _, row in sub.iterrows():
    pred = str(row["PredictionString"]).strip()

    if pred == "":
        continue

    vals = pred.split()
    if len(vals) % 6 != 0:
        valid = False
        bad_reason = f"Wrong number of values for {row['image_id']}"
        break

    boxes = np.array(vals, dtype=float).reshape(-1, 6)

    if not np.isfinite(boxes).all():
        valid = False
        bad_reason = f"NaN/Inf in {row['image_id']}"
        break

    if not np.all((boxes[:, 0] >= 0) & (boxes[:, 0] < 8)):
        valid = False
        bad_reason = f"Bad class id in {row['image_id']}"
        break

    if not np.all(boxes[:, 0] == np.floor(boxes[:, 0])):
        valid = False
        bad_reason = f"Non-integer class id in {row['image_id']}"
        break

    if not np.all((boxes[:, 1] >= 0) & (boxes[:, 1] <= 1)):
        valid = False
        bad_reason = f"Bad confidence in {row['image_id']}"
        break

    if not np.all(boxes[:, 4] > 0):
        valid = False
        bad_reason = f"Non-positive width in {row['image_id']}"
        break

    if not np.all(boxes[:, 5] > 0):
        valid = False
        bad_reason = f"Non-positive height in {row['image_id']}"
        break

print("Prediction format valid:", valid)
print("Reason:", bad_reason)
print("Ready for Kaggle:", valid)

Submission shape: (3811, 2)
Unique image ids: 3811
Expected test rows: 3811
Missing IDs: 0
Extra IDs: 0
Prediction format valid: True
Reason: None
Ready for Kaggle: True
